In [0]:
# Databricks notebook source
def path_exists(path):
  try:
    dbutils.fs.ls(path)
    return True
  except Exception as e:
    if 'java.io.FileNotFoundException' in str(e):
      return False
    raise

# COMMAND ----------

def download_dataset(source, target):
    # Ensure target directory exists
    dbutils.fs.mkdirs(target)
    
    files = dbutils.fs.ls(source)
    for f in files:
        if f.name.endswith('/'):  # Skip directory entries
            continue
        source_path = f.path  # Use .path instead of string concatenation
        target_path = f"{target.rstrip('/')}/{f.name}"
        if not path_exists(target_path):
            print(f"Copying {f.name}...")
            dbutils.fs.cp(source_path, target_path, True)

# COMMAND ----------

# Configure with your actual S3 bucket and paths
data_source_uri = "s3://dalhussein-courses/datasets/bookstore/v1/"
dataset_bookstore = '/mnt/demo-datasets/bookstore'  # Removed 'dbfs:' prefix
data_catalog = 'hive_metastore'

# Mount S3 bucket if not already mounted (run once)
try:
    dbutils.fs.ls(dataset_bookstore)
except:
    dbutils.fs.mount(f"s3a://dalhussein-courses", "/mnt/demo-datasets",
                    extra_configs={"fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider"})

spark.conf.set("dataset.bookstore", dataset_bookstore)

# COMMAND ----------

def get_index(dir):
    try:
        files = [f for f in dbutils.fs.ls(dir) if not f.name.endswith('/')]
        if files:
            file = max(files, key=lambda x: x.name).name
            return int(file.split('.')[0]) + 1
        return 1
    except:
        return 1

# COMMAND ----------

def set_current_catalog(catalog_name):
    spark.sql(f"USE CATALOG {catalog_name}")

# COMMAND ----------

# Structured Streaming
streaming_dir = f"{dataset_bookstore}/orders-streaming"
raw_dir = f"{dataset_bookstore}/orders-raw"

def load_file(current_index):
    latest_file = f"{str(current_index).zfill(2)}.parquet"
    print(f"Loading {latest_file} to {raw_dir}")
    dbutils.fs.cp(f"{streaming_dir}/{latest_file}", f"{raw_dir}/{latest_file}")

def load_new_data(all=False):
    index = get_index(raw_dir)
    if index > 10:
        print("No more data to load")
        return
        
    if all:
        while index <= 10:
            load_file(index)
            index += 1
    else:
        load_file(index)

# COMMAND ----------

# DLT Pipeline Functions
streaming_orders_dir = f"{dataset_bookstore}/orders-json-streaming"
streaming_books_dir = f"{dataset_bookstore}/books-streaming"
raw_orders_dir = f"{dataset_bookstore}/orders-json-raw"
raw_books_dir = f"{dataset_bookstore}/books-cdc"

def load_json_file(current_index):
    latest_file = f"{str(current_index).zfill(2)}.json"
    print(f"Copying {latest_file} to orders and books dirs")
    dbutils.fs.cp(f"{streaming_orders_dir}/{latest_file}", f"{raw_orders_dir}/{latest_file}")
    dbutils.fs.cp(f"{streaming_books_dir}/{latest_file}", f"{raw_books_dir}/{latest_file}")

def load_new_json_data(all=False):
    index = get_index(raw_orders_dir)
    if index > 10:
        print("No more data to load")
        return
        
    if all:
        while index <= 10:
            load_json_file(index)
            index += 1
    else:
        load_json_file(index)

# COMMAND ----------

# Execute the download
download_dataset(data_source_uri, dataset_bookstore)
set_current_catalog(data_catalog)

# Verify download
display(dbutils.fs.ls(dataset_bookstore))